# FinMark Corporation — Milestone 1: ML Solution EDA & Data Preprocessing

**Business problem:** FinMark Corporation has gathered diverse datasets — customer transactions,
social media interactions, and demographic profiles — but struggles to derive actionable insights
because of data volume and inconsistencies across sources. This limits their ability to spot
market trends, understand customer behavior, and make data-driven decisions.

**This notebook covers:**
1. Loading the three raw datasets into pandas DataFrames
2. Initial inspection (duplicates, missing values, data types)
3. Preprocessing / cleaning each dataset
4. Validating the cleaned output
5. Saving finalized, analysis-ready datasets

**Datasets:**
- `customer_demographics_contaminated.csv`
- `customer_transactions_contaminated.csv`
- `social_media_interactions_contaminated.csv`


## 1. Setup & Load Datasets

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

DATA_DIR = ''  # datasets are in the same folder as this notebook

demographics = pd.read_csv(DATA_DIR + 'customer_demographics_contaminated.csv')
transactions = pd.read_csv(DATA_DIR + 'customer_transactions_contaminated.csv')
social = pd.read_csv(DATA_DIR + 'social_media_interactions_contaminated.csv')

print('Demographics :', demographics.shape)
print('Transactions :', transactions.shape)
print('Social media :', social.shape)


Demographics : (3200, 6)
Transactions : (3200, 6)
Social media : (3200, 6)


In [2]:
demographics.head()

,CustomerID,Age,Gender,Location,IncomeLevel,SignupDate
0,9207fa75-5758-48d1-94ad-19c041e0520f,51.0,Female,Jensenberg,Low,2022-11-17
1,5fb09cd8-a473-46f7-80bd-6e49cf509078,NaN,Female,Castilloport,High,2020-07-21
2,c139496e-cc89-498a-bd90-1fb4627b6cff,37.0,Male,Lake Jennifertown,NaN,2021-01-01
3,50118139-7264-428f-81cc-a25fddc5d6dd,44.0,Male,Port Carl,Medium,2024-06-10
4,7d1f2bbc-8d16-4fbc-9b37-ece3324e8ed4,50.0,Female,Jessebury,High,2023-08-24


In [3]:
transactions.head()

,CustomerID,TransactionID,TransactionDate,Amount,ProductCategory,PaymentMethod
0,60567026-f719-4cd6-849e-137e86d8938f,5ff75116-0a50-4d04-80fb-31e5ccbb0769,2024-05-15,117.64,Clothing,PayPal
1,4090ba85-b111-4f75-a792-c777965f5255,2c39b9fe-ff57-4d39-9321-9f5cdf187aa1,2023-04-26,466.14,Health & Beauty,Bank Transfer
2,9223891b-73ff-4d5c-b8ae-13ece82ee28b,f79588dd-3db9-4ffa-97f8-7de0e64259f1,2022-09-23,563.99,Clothing,Debit Card
3,9243eebc-938f-480c-8564-16d503d250de,401c0fc9-60df-4455-ad78-67c132f9897d,2024-04-15,254.44,Automotive,PayPal
4,6e3e8eb8-bc0f-4ffe-9f74-5d5efec9502f,2034aebc-8280-4254-a667-92bcd1c2be4f,2024-06-03,590.52,Home & Garden,Bank Transfer


In [4]:
social.head()

,CustomerID,InteractionID,InteractionDate,Platform,InteractionType,Sentiment
0,2dcb9523-356b-40b2-a67b-1f27797de261,e5d15761-d0a7-4329-89e3-79a892c56097,2023-07-11,NaN,Comment,NaN
1,e12c37b3-7d4d-472f-9fd8-0df2cb3001aa,02f9f376-70ae-4fcd-9070-1db977939948,2023-07-06,Twitter,Share,NaN
2,08a911a3-65e6-4f5d-a6a1-ae7ddcbe28a2,a83fa04c-f109-4f24-8ce1-2078154f6a1c,2024-05-24,Instagram,Comment,Neutral
3,efdfdfc9-5dbb-4478-911a-101a390a0285,28a69c4b-a2e4-4c74-a130-1132d7733fdf,2023-11-01,Instagram,Like,Neutral
4,ca1e90f6-0e5f-492e-ab92-252ff540da18,d9d1c6f8-5e15-4738-b52b-13c2982420cc,2023-07-08,Instagram,Like,NaN


## 2. Initial Inspection

For each dataset we check:
- Data types of every column
- Duplicate rows
- Missing values per column


### 2.1 Customer Demographics

In [5]:
print(demographics.dtypes)
print()
print('Full duplicate rows:', demographics.duplicated().sum())
print('Duplicate CustomerIDs:', demographics['CustomerID'].duplicated().sum())
print()
print('Missing values per column:')
print(demographics.isna().sum())


CustomerID     str
Age            str
Gender         str
Location       str
IncomeLevel    str
SignupDate     str
dtype: object

Full duplicate rows: 177
Duplicate CustomerIDs: 200

Missing values per column:
CustomerID       0
Age            291
Gender           0
Location         0
IncomeLevel    303
SignupDate       0
dtype: int64


In [6]:
# Look for hidden/contaminated values in Age (should be numeric, plausible range)
print(sorted(demographics['Age'].dropna().unique(), key=str)[:10], '...')
print('Non-numeric Age values:', [v for v in demographics['Age'].dropna().unique()
                                   if not str(v).replace('.', '', 1).replace('-', '', 1).isdigit()])
age_numeric = pd.to_numeric(demographics['Age'], errors='coerce')
print('Age values outside plausible 0-100 range:', sorted(age_numeric[(age_numeric < 0) | (age_numeric > 100)].unique()))


['-1', '-1.0', '150', '150.0', '18.0', '19.0', '20.0', '21.0', '22.0', '23.0'] ...
Non-numeric Age values: ['Unknown']
Age values outside plausible 0-100 range: [np.float64(-1.0), np.float64(150.0)]


In [7]:
# Signup date format check - is it consistent?
import re
def classify_date(v):
    v = str(v)
    if re.match(r'^\d{4}-\d{2}-\d{2}$', v):
        return 'YYYY-MM-DD'
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', v):
        return 'DD/MM/YYYY'
    return 'OTHER'

print(demographics['SignupDate'].apply(classify_date).value_counts())


SignupDate
YYYY-MM-DD    3102
DD/MM/YYYY      98
Name: count, dtype: int64


**Findings — Demographics:**
- `Age` contains sentinel/placeholder junk values: `"Unknown"`, `-1`, `-1.0`, `150`, `150.0` — not real ages.
- `Age` has 291 true missing values (`NaN`).
- `IncomeLevel` has 303 missing values.
- `SignupDate` is stored in two different formats (`YYYY-MM-DD` and `DD/MM/YYYY`) — needs normalization before it can be parsed as a single date type.
- 177 fully duplicated rows, and 200 duplicated `CustomerID`s in total (some of those 200 have a `SignupDate` in a different format but otherwise the same info — i.e. they're the *same* duplicate once dates are normalized).


### 2.2 Customer Transactions

In [8]:
print(transactions.dtypes)
print()
print('Full duplicate rows:', transactions.duplicated().sum())
print('Duplicate TransactionIDs:', transactions['TransactionID'].duplicated().sum())
print()
print('Missing values per column:')
print(transactions.isna().sum())


CustomerID         str
TransactionID      str
TransactionDate    str
Amount             str
ProductCategory    str
PaymentMethod      str
dtype: object

Full duplicate rows: 185
Duplicate TransactionIDs: 200

Missing values per column:
CustomerID           0
TransactionID        0
TransactionDate      0
Amount             304
ProductCategory    299
PaymentMethod        0
dtype: int64


In [9]:
# Amount should be numeric - check for contamination
print('Non-numeric Amount values:', transactions.loc[pd.to_numeric(transactions['Amount'], errors='coerce').isna()
                                                       & transactions['Amount'].notna(), 'Amount'].unique())
amt_numeric = pd.to_numeric(transactions['Amount'], errors='coerce')
print('Negative Amount value counts:')
print(amt_numeric[amt_numeric < 0].value_counts())
print('Amount == 0 count:', (amt_numeric == 0).sum())


Non-numeric Amount values: <StringArray>
['Free']
Length: 1, dtype: str
Negative Amount value counts:
Amount
-100.0    40
Name: count, dtype: int64
Amount == 0 count: 30


In [10]:
print(transactions['ProductCategory'].unique())
print(transactions['PaymentMethod'].unique())
print(transactions['TransactionDate'].apply(classify_date).value_counts())


<StringArray>
['Clothing', 'Health & Beauty', 'Automotive', 'Home & Garden', nan, 'Electronics']
Length: 6, dtype: str
<StringArray>
['PayPal', 'Bank Transfer', 'Debit Card', 'Credit Card']
Length: 4, dtype: str
TransactionDate
YYYY-MM-DD    3102
DD/MM/YYYY      98
Name: count, dtype: int64


**Findings — Transactions:**
- `Amount` contains the text value `"Free"` (3 rows) instead of a number, and a sentinel value of exactly **`-100`** in 40 rows — a suspiciously uniform number that doesn't look like a real negative charge, more like an error code.
- `Amount` has 304 - 43 (above) = 261 true `NaN` values.
- `ProductCategory` has 299 missing values; the categories themselves (`Clothing`, `Health & Beauty`, `Automotive`, `Home & Garden`, `Electronics`) are otherwise clean and consistently spelled.
- `PaymentMethod` has no missing values and clean categories.
- `TransactionDate` has the same two-format problem as `SignupDate`.
- 185 fully duplicated rows. Of the 200 `TransactionID`s that appear twice, 185 pairs are exact duplicates and 15 pairs share the same ID but have *different* data in other columns — a genuine ID-collision data quality issue.


### 2.3 Social Media Interactions

In [11]:
print(social.dtypes)
print()
print('Full duplicate rows:', social.duplicated().sum())
print('Duplicate InteractionIDs:', social['InteractionID'].duplicated().sum())
print()
print('Missing values per column:')
print(social.isna().sum())


CustomerID         str
InteractionID      str
InteractionDate    str
Platform           str
InteractionType    str
Sentiment          str
dtype: object

Full duplicate rows: 180
Duplicate InteractionIDs: 200

Missing values per column:
CustomerID           0
InteractionID        0
InteractionDate      0
Platform           311
InteractionType      0
Sentiment          329
dtype: int64


In [12]:
print(social['Platform'].unique())
print(social['InteractionType'].unique())
print(social['Sentiment'].unique())
print(social['InteractionDate'].apply(classify_date).value_counts())


<StringArray>
[nan, 'Twitter', 'Instagram', 'Facebook']
Length: 4, dtype: str
<StringArray>
['Comment', 'Share', 'Like']
Length: 3, dtype: str
<StringArray>
[nan, 'Neutral', 'Positive', 'Negative', 'Very Negative', 'Very Positive']
Length: 6, dtype: str
InteractionDate
YYYY-MM-DD    3100
DD/MM/YYYY     100
Name: count, dtype: int64


**Findings — Social Media Interactions:**
- `Platform` has 311 missing values; categories are otherwise clean (`Twitter`, `Instagram`, `Facebook`).
- `Sentiment` has 329 missing values; categories are otherwise clean (`Neutral`, `Positive`, `Negative`, `Very Positive`, `Very Negative`).
- `InteractionType` has no missing values and clean categories (`Comment`, `Share`, `Like`).
- `InteractionDate` has the same two-format problem.
- 180 fully duplicated rows; 20 pairs share an `InteractionID` with conflicting data (same collision issue as transactions).


## 3. Preprocessing

General approach used across all three datasets:
1. Normalize mixed date formats to a single `datetime64` type.
2. Replace contamination sentinels (e.g. `Unknown`, `-1`, `150`, `-100`, `Free`) with proper `NaN` / real values, since they are not valid data — they're placeholder/error values.
3. Drop exact duplicate rows.
4. Resolve duplicate IDs that carry conflicting data (keep the more complete record).
5. Impute or flag remaining missing values in a way that suits each column's role (numeric vs. categorical).
6. Enforce final, correct data types.


### 3.1 Clean Customer Demographics

In [13]:
demo_clean = demographics.copy()

# --- Normalize SignupDate (two formats -> single datetime) ---
def parse_mixed_date(v):
    d1 = pd.to_datetime(v, format='%Y-%m-%d', errors='coerce')
    if pd.isna(d1):
        d1 = pd.to_datetime(v, dayfirst=True, errors='coerce')
    return d1

demo_clean['SignupDate'] = demo_clean['SignupDate'].apply(parse_mixed_date)

# --- Clean Age: strip sentinel junk values, coerce to numeric ---
demo_clean['Age'] = demo_clean['Age'].replace(['Unknown'], np.nan)
demo_clean['Age'] = pd.to_numeric(demo_clean['Age'], errors='coerce')
demo_clean.loc[(demo_clean['Age'] < 0) | (demo_clean['Age'] > 100), 'Age'] = np.nan

# --- Drop exact duplicate rows first ---
demo_clean = demo_clean.drop_duplicates()

# --- Resolve remaining duplicate CustomerIDs: keep the most complete record per customer ---
demo_clean['_completeness'] = demo_clean.notna().sum(axis=1)
demo_clean = (demo_clean
              .sort_values('_completeness', ascending=False)
              .drop_duplicates(subset='CustomerID', keep='first')
              .drop(columns='_completeness')
              .sort_index())

# --- Impute remaining missing values ---
demo_clean['Age'] = demo_clean['Age'].fillna(demo_clean['Age'].median())
demo_clean['IncomeLevel'] = demo_clean['IncomeLevel'].fillna('Unknown')

# --- Final dtypes ---
demo_clean['Age'] = demo_clean['Age'].round().astype('Int64')
demo_clean['Gender'] = demo_clean['Gender'].astype('category')
demo_clean['IncomeLevel'] = demo_clean['IncomeLevel'].astype('category')

demo_clean = demo_clean.reset_index(drop=True)
demo_clean.info()


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   CustomerID   3000 non-null   str           
 1   Age          3000 non-null   Int64         
 2   Gender       3000 non-null   category      
 3   Location     3000 non-null   str           
 4   IncomeLevel  3000 non-null   category      
 5   SignupDate   3000 non-null   datetime64[us]
dtypes: Int64(1), category(2), datetime64[us](1), str(2)
memory usage: 102.7 KB


In [14]:
demo_clean.head()

,CustomerID,Age,Gender,Location,IncomeLevel,SignupDate
0,9207fa75-5758-48d1-94ad-19c041e0520f,51,Female,Jensenberg,Low,2022-11-17
1,5fb09cd8-a473-46f7-80bd-6e49cf509078,45,Female,Castilloport,High,2020-07-21
2,c139496e-cc89-498a-bd90-1fb4627b6cff,37,Male,Lake Jennifertown,Unknown,2021-01-01
3,50118139-7264-428f-81cc-a25fddc5d6dd,44,Male,Port Carl,Medium,2024-06-10
4,7d1f2bbc-8d16-4fbc-9b37-ece3324e8ed4,50,Female,Jessebury,High,2023-08-24


### 3.2 Clean Customer Transactions

In [15]:
txn_clean = transactions.copy()

# --- Normalize TransactionDate ---
txn_clean['TransactionDate'] = txn_clean['TransactionDate'].apply(parse_mixed_date)

# --- Clean Amount: 'Free' -> 0, sentinel -100 -> NaN, coerce numeric ---
txn_clean['Amount'] = txn_clean['Amount'].replace({'Free': 0})
txn_clean['Amount'] = pd.to_numeric(txn_clean['Amount'], errors='coerce')
txn_clean.loc[txn_clean['Amount'] == -100, 'Amount'] = np.nan

# --- Drop exact duplicate rows first ---
txn_clean = txn_clean.drop_duplicates()

# --- Resolve remaining duplicate TransactionIDs (ID collisions): keep most complete record ---
txn_clean['_completeness'] = txn_clean.notna().sum(axis=1)
txn_clean = (txn_clean
             .sort_values('_completeness', ascending=False)
             .drop_duplicates(subset='TransactionID', keep='first')
             .drop(columns='_completeness')
             .sort_index())

# --- Impute remaining missing values ---
txn_clean['Amount'] = txn_clean['Amount'].fillna(txn_clean['Amount'].median())
txn_clean['ProductCategory'] = txn_clean['ProductCategory'].fillna('Unknown')

# --- Final dtypes ---
txn_clean['Amount'] = txn_clean['Amount'].round(2)
txn_clean['ProductCategory'] = txn_clean['ProductCategory'].astype('category')
txn_clean['PaymentMethod'] = txn_clean['PaymentMethod'].astype('category')

txn_clean = txn_clean.reset_index(drop=True)
txn_clean.info()


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   CustomerID       3000 non-null   str           
 1   TransactionID    3000 non-null   str           
 2   TransactionDate  3000 non-null   datetime64[us]
 3   Amount           3000 non-null   float64       
 4   ProductCategory  3000 non-null   category      
 5   PaymentMethod    3000 non-null   category      
dtypes: category(2), datetime64[us](1), float64(1), str(2)
memory usage: 99.8 KB


In [16]:
txn_clean.head()

,CustomerID,TransactionID,TransactionDate,Amount,ProductCategory,PaymentMethod
0,60567026-f719-4cd6-849e-137e86d8938f,5ff75116-0a50-4d04-80fb-31e5ccbb0769,2024-05-15,117.64,Clothing,PayPal
1,4090ba85-b111-4f75-a792-c777965f5255,2c39b9fe-ff57-4d39-9321-9f5cdf187aa1,2023-04-26,466.14,Health & Beauty,Bank Transfer
2,9223891b-73ff-4d5c-b8ae-13ece82ee28b,f79588dd-3db9-4ffa-97f8-7de0e64259f1,2022-09-23,563.99,Clothing,Debit Card
3,9243eebc-938f-480c-8564-16d503d250de,401c0fc9-60df-4455-ad78-67c132f9897d,2024-04-15,254.44,Automotive,PayPal
4,6e3e8eb8-bc0f-4ffe-9f74-5d5efec9502f,2034aebc-8280-4254-a667-92bcd1c2be4f,2024-06-03,590.52,Home & Garden,Bank Transfer


### 3.3 Clean Social Media Interactions

In [17]:
social_clean = social.copy()

# --- Normalize InteractionDate ---
social_clean['InteractionDate'] = social_clean['InteractionDate'].apply(parse_mixed_date)

# --- Drop exact duplicate rows first ---
social_clean = social_clean.drop_duplicates()

# --- Resolve remaining duplicate InteractionIDs (ID collisions): keep most complete record ---
social_clean['_completeness'] = social_clean.notna().sum(axis=1)
social_clean = (social_clean
                .sort_values('_completeness', ascending=False)
                .drop_duplicates(subset='InteractionID', keep='first')
                .drop(columns='_completeness')
                .sort_index())

# --- Impute remaining missing values (categorical -> explicit 'Unknown') ---
social_clean['Platform'] = social_clean['Platform'].fillna('Unknown')
social_clean['Sentiment'] = social_clean['Sentiment'].fillna('Unknown')

# --- Final dtypes ---
social_clean['Platform'] = social_clean['Platform'].astype('category')
social_clean['InteractionType'] = social_clean['InteractionType'].astype('category')
social_clean['Sentiment'] = social_clean['Sentiment'].astype('category')

social_clean = social_clean.reset_index(drop=True)
social_clean.info()


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   CustomerID       3000 non-null   str           
 1   InteractionID    3000 non-null   str           
 2   InteractionDate  3000 non-null   datetime64[us]
 3   Platform         3000 non-null   category      
 4   InteractionType  3000 non-null   category      
 5   Sentiment        3000 non-null   category      
dtypes: category(3), datetime64[us](1), str(2)
memory usage: 79.3 KB


In [18]:
social_clean.head()

,CustomerID,InteractionID,InteractionDate,Platform,InteractionType,Sentiment
0,2dcb9523-356b-40b2-a67b-1f27797de261,e5d15761-d0a7-4329-89e3-79a892c56097,2023-07-11,Unknown,Comment,Unknown
1,e12c37b3-7d4d-472f-9fd8-0df2cb3001aa,02f9f376-70ae-4fcd-9070-1db977939948,2023-07-06,Twitter,Share,Unknown
2,08a911a3-65e6-4f5d-a6a1-ae7ddcbe28a2,a83fa04c-f109-4f24-8ce1-2078154f6a1c,2024-05-24,Instagram,Comment,Neutral
3,efdfdfc9-5dbb-4478-911a-101a390a0285,28a69c4b-a2e4-4c74-a130-1132d7733fdf,2023-11-01,Instagram,Like,Neutral
4,ca1e90f6-0e5f-492e-ab92-252ff540da18,d9d1c6f8-5e15-4738-b52b-13c2982420cc,2023-07-08,Instagram,Like,Unknown


## 4. Post-Cleaning Validation

Re-run the same checks used during initial inspection to confirm the cleaning worked.


In [19]:
for name, df, id_col in [('Demographics', demo_clean, 'CustomerID'),
                          ('Transactions', txn_clean, 'TransactionID'),
                          ('Social', social_clean, 'InteractionID')]:
    print(f'--- {name} ---')
    print('Shape:', df.shape)
    print('Full duplicate rows:', df.duplicated().sum())
    print('Duplicate IDs:', df[id_col].duplicated().sum())
    print('Missing values:\n', df.isna().sum())
    print()


--- Demographics ---
Shape: (3000, 6)
Full duplicate rows: 0
Duplicate IDs: 0
Missing values:
 CustomerID     0
Age            0
Gender         0
Location       0
IncomeLevel    0
SignupDate     0
dtype: int64

--- Transactions ---
Shape: (3000, 6)
Full duplicate rows: 0
Duplicate IDs: 0
Missing values:
 CustomerID         0
TransactionID      0
TransactionDate    0
Amount             0
ProductCategory    0
PaymentMethod      0
dtype: int64

--- Social ---
Shape: (3000, 6)
Full duplicate rows: 0
Duplicate IDs: 0
Missing values:
 CustomerID         0
InteractionID      0
InteractionDate    0
Platform           0
InteractionType    0
Sentiment          0
dtype: int64



In [20]:
# Sanity check numeric ranges after cleaning
print('Age range:', demo_clean['Age'].min(), '-', demo_clean['Age'].max())
print('Amount range:', txn_clean['Amount'].min(), '-', txn_clean['Amount'].max())
print('SignupDate range:', demo_clean['SignupDate'].min(), '-', demo_clean['SignupDate'].max())
print('TransactionDate range:', txn_clean['TransactionDate'].min(), '-', txn_clean['TransactionDate'].max())
print('InteractionDate range:', social_clean['InteractionDate'].min(), '-', social_clean['InteractionDate'].max())


Age range: 18 - 70
Amount range: 0.0 - 999.86
SignupDate range: 2019-07-01 00:00:00 - 2024-06-30 00:00:00
TransactionDate range: 2022-07-01 00:00:00 - 2024-06-30 00:00:00
InteractionDate range: 2023-07-01 00:00:00 - 2024-06-30 00:00:00


## 5. Save Finalized Preprocessed Datasets

In [21]:
demo_clean.to_csv('customer_demographics_cleaned.csv', index=False)
txn_clean.to_csv('customer_transactions_cleaned.csv', index=False)
social_clean.to_csv('social_media_interactions_cleaned.csv', index=False)

print('Saved:')
print(' - customer_demographics_cleaned.csv       ', demo_clean.shape)
print(' - customer_transactions_cleaned.csv        ', txn_clean.shape)
print(' - social_media_interactions_cleaned.csv    ', social_clean.shape)


Saved:
 - customer_demographics_cleaned.csv        (3000, 6)
 - customer_transactions_cleaned.csv         (3000, 6)
 - social_media_interactions_cleaned.csv     (3000, 6)


## 6. Summary of Preprocessing Decisions (for team discussion)

**Preprocessing steps performed**
1. Parsed two inconsistent date formats (`YYYY-MM-DD` and `DD/MM/YYYY`) into a single `datetime64` type in all three files.
2. Removed exact duplicate rows (demographics: 177, transactions: 185, social: 180).
3. Resolved duplicate primary/foreign-key IDs that had conflicting field values by keeping the most complete record per ID, since we can't be sure which copy is authoritative and dropping the sparser one loses the least information.
4. Replaced contamination sentinels with proper missing values:
   - `Age`: `"Unknown"`, `-1`, `150` → `NaN` (impossible ages).
   - `Amount`: `-100` → `NaN` (suspiciously uniform, most likely an error code, not a real refund).
   - `Amount`: `"Free"` → `0` (a legitimate zero-cost transaction, not a data error).
5. Imputed remaining missing values:
   - Numeric (`Age`, `Amount`) → column median, to avoid distorting the distribution with outlier-sensitive means.
   - Categorical (`IncomeLevel`, `ProductCategory`, `Platform`, `Sentiment`) → explicit `"Unknown"` category rather than the mode, to avoid manufacturing false signal.
6. Cast final columns to appropriate dtypes (`category` for categoricals, nullable `Int64` for Age, `datetime64` for all date columns).

**Challenges encountered**
- Distinguishing genuine missing values from *disguised* missing values (sentinels like `-1`, `150`, `-100`, `"Unknown"`) required manually profiling each column's unique values rather than trusting `isna()` alone.
- Duplicate IDs weren't always exact duplicates — some carried conflicting data, which meant a rule had to be chosen (most-complete-record) rather than a simple `drop_duplicates()`.
- Mixed date formats meant a naive `pd.to_datetime()` call silently produced `NaT` or misparsed values for one of the two formats.

**Decisions made for handling missing/inconsistent data**
- Prefer median over mean for numeric imputation given skew/outlier risk.
- Prefer an explicit `"Unknown"` category over mode-imputation for categoricals, so downstream models don't get a false confidence signal about missing customer attributes.
- Treated `"Free"` as real (0) but treated `-100` as invalid, based on plausibility (a uniform negative constant looks like an error flag, not real transaction data).

**Observed patterns / data quality issues worth flagging to the team**
- All three source files appear to have been deliberately contaminated with duplicate rows, disguised missing values (sentinels), and inconsistent date formats — consistent with a synthetic "dirty data" exercise.
- Duplicate rate is consistent across all three files (~5.5–6%), suggesting the contamination was applied uniformly.
- `CustomerID` is the natural join key across all three datasets and should be validated for referential integrity (i.e., do all transaction/social `CustomerID`s exist in demographics?) as part of the next EDA milestone.
